<a href="https://colab.research.google.com/github/pavlenkosasha/MillionaireAPI/blob/main/Uber_Pickups_in_New_York_City_PAVLENKO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

*Что мы подключили
pandas — работа с таблицами и CSV;
numpy — работа с числами и массивами;
matplotlib — построение графиков;*текст курсивом*
seaborn — удобная визуализация данных поверх Matplotlib.*

**Загружаем CSV**

In [2]:
from google.colab import files

uploaded = files.upload()

Saving archive.zip to archive.zip


In [3]:
import zipfile

with zipfile.ZipFile("archive.zip", "r") as zip_ref:
    print(zip_ref.namelist())

['Uber-Jan-Feb-FOIL.csv', 'other-American_B01362.csv', 'other-Carmel_B00256.csv', 'other-Dial7_B00887.csv', 'other-Diplo_B01196.csv', 'other-FHV-services_jan-aug-2015.csv', 'other-Federal_02216.csv', 'other-Firstclass_B01536.csv', 'other-Highclass_B01717.csv', 'other-Lyft_B02510.csv', 'other-Prestige_B01338.csv', 'other-Skyline_B00111.csv', 'uber-raw-data-apr14.csv', 'uber-raw-data-aug14.csv', 'uber-raw-data-janjune-15.csv', 'uber-raw-data-jul14.csv', 'uber-raw-data-jun14.csv', 'uber-raw-data-may14.csv', 'uber-raw-data-sep14.csv']


**завантажую лише  1000 поїздок,(для навчального проеку)**

In [4]:
import zipfile
import pandas as pd

with zipfile.ZipFile("archive.zip", "r") as zip_ref:
    with zip_ref.open("uber-raw-data-apr14.csv") as file:
        df = pd.read_csv(file, nrows=1000)

print("Кількість поїздок:", len(df))
print("Розмір таблиці:", df.shape)

Кількість поїздок: 1000
Розмір таблиці: (1000, 4)


**переглядаємо дані**

In [6]:
df.head()

,Date/Time,Lat,Lon,Base
0,4/1/2014 0:11:00,40.7690,-73.9549,B02512
1,4/1/2014 0:17:00,40.7267,-74.0345,B02512
2,4/1/2014 0:21:00,40.7316,-73.9873,B02512
3,4/1/2014 0:28:00,40.7588,-73.9776,B02512
4,4/1/2014 0:33:00,40.7594,-73.9722,B02512


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Date/Time  1000 non-null   object 
 1   Lat        1000 non-null   float64
 2   Lon        1000 non-null   float64
 3   Base       1000 non-null   object 
dtypes: float64(2), object(2)
memory usage: 31.4+ KB


 df.describe()

*Ця команда покаже статистику для числових колонок:

count — кількість значень;
mean — середнє значення;
std — стандартне відхилення;
min — мінімальне значення;
25% — перший квартиль;
50% — медіана;
75% — третій квартиль;
max — максимальне значення.*

In [8]:
df.describe()

,Lat,Lon
count,1000.000000,1000.000000
mean,40.748463,-73.983861
std,0.034128,0.062241
min,40.608600,-74.420000
25%,40.728400,-73.999125
50%,40.753750,-73.983600
75%,40.767325,-73.969875
max,40.985900,-73.420200


In [9]:
df.isnull().sum()

,0
Date/Time,0
Lat,0
Lon,0
Base,0


перевірили пропушені значення
У наших перших 1000 поїздках немає жодного пропущеного значення. ✅

Тобто всі 1000 рядків мають:

дату й час;
широту Lat;
довготу Lon;
Base.

**перевірим дублікати**

In [10]:
df.duplicated().sum()

np.int64(33)

In [11]:
df[df.duplicated()]

,Date/Time,Lat,Lon,Base
44,4/1/2014 5:44:00,40.7430,-74.0301,B02512
128,4/1/2014 7:25:00,40.7805,-73.9481,B02512
185,4/1/2014 8:25:00,40.7620,-73.9787,B02512
190,4/1/2014 8:29:00,40.6904,-74.1778,B02512
238,4/1/2014 9:49:00,40.7195,-74.0367,B02512
265,4/1/2014 10:32:00,40.6949,-74.1781,B02512
291,4/1/2014 11:01:00,40.7883,-74.0456,B02512
299,4/1/2014 11:06:00,40.8275,-74.0468,B02512
345,4/1/2014 12:02:00,40.7852,-74.0220,B02512
349,4/1/2014 12:10:00,40.8289,-73.9451,B02512


перевіримо, скільки у нас унікальних поїздок

In [12]:
df.nunique()

,0
Date/Time,644
Lat,592
Lon,582
Base,1


In [13]:
df["Base"].unique()

array(['B02512'], dtype=object)

*Пізніше, на етапі підготовки даних, ми, скоріш за все, видалимо Base, оскільки вона має тільки одне унікальне значення.*


*перетворимо Date/Time із тексту в справжній формат дати й часу.*

In [14]:
df["Date/Time"] = pd.to_datetime(df["Date/Time"])

*створимо окрему колонку Hour — година, коли було замовлення.*

In [15]:
df["Hour"] = df["Date/Time"].dt.hour